# Concurrent Execution with Fleche

This notebook shows how to use `@fleche`-decorated functions with Python's two standard executor pools:

| Executor | Cache context propagation | How |
|---|---|---|
| `ThreadPoolExecutor` | ✅ Yes (with one trick) | `contextvars.copy_context().run` |
| `ProcessPoolExecutor` | ⚠️ Manual setup required | Re-create cache inside each worker |

In [ ]:
import time
import contextvars
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from fleche import fleche, cache
from fleche.caches import Cache
from fleche.storage.memory import Memory

## 1. ThreadPoolExecutor

### The problem: threads don't inherit the cache context by default

Python threads each start with a *copy* of the context that existed when `ThreadPoolExecutor` was created, not a live reference to the calling thread's context. This means that if you set a cache inside the main thread and submit work to a pool, the workers won't see it.

In [ ]:
@fleche
def add(x, y):
    return x + y

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    with ThreadPoolExecutor(max_workers=2) as pool:
        pool.submit(add, 1, 2).result()

# The result was NOT stored in my_cache — the thread used the default cache
print("In my_cache:", my_cache.contains(add.digest(1, 2)))  # False

### The fix: propagate context explicitly with `ctx.run`

Capture the current context with `contextvars.copy_context()` and wrap the call in `ctx.run(...)`. This hands the *exact same context object* to the thread, so the `cache(...)` context manager is visible inside the worker.

In [ ]:
mem = Memory({})
my_cache = Cache(mem, mem)

with cache(my_cache):
    ctx = contextvars.copy_context()          # snapshot the current context
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [
            pool.submit(ctx.run, add, x, x + 1)  # ctx.run propagates the context
            for x in range(4)
        ]
        results = [f.result() for f in futures]

print("Results:", results)                       # [1, 3, 5, 7]
print("In my_cache:", my_cache.contains(add.digest(0, 1)))  # True

### Shared cache across all threads

Because all workers run inside the same context, they share the same in-memory store. Results computed by one thread are immediately visible to every other thread in the pool.

In [ ]:
@fleche
def slow_square(x):
    time.sleep(0.1)   # simulate work
    return x * x

mem = Memory({})
my_cache = Cache(mem, mem)

# First batch — computes and caches
with cache(my_cache):
    ctx = contextvars.copy_context()
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(lambda x: ctx.run(slow_square, x), range(5)))

# Second batch — everything hits the cache (no sleep)
start = time.time()
with cache(my_cache):
    ctx = contextvars.copy_context()
    with ThreadPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(lambda x: ctx.run(slow_square, x), range(5)))

elapsed = time.time() - start
print(f"Results: {results}")
print(f"Second batch took {elapsed:.3f}s (cache hits — no sleep)")

---
## 2. ProcessPoolExecutor

### Why `ctx.run` doesn't work across processes

With `ProcessPoolExecutor`, every argument passed to a worker must be serialised with `pickle` and sent to a new Python interpreter. `contextvars.Context` objects **cannot be pickled**, so the threadpool trick is not available:

```python
# ❌ This raises: TypeError: cannot pickle 'Context' object
ctx = contextvars.copy_context()
executor.submit(ctx.run, add, 1, 2)
```

The good news: `@fleche`-decorated functions *are* picklable (they are module-level names), so you can pass them as worker targets without any issues.

### The pattern: set up cache inside the worker

Each worker process starts with a fresh Python interpreter that has no cache configured. The simplest fix is to wrap your cached call inside a thin helper that creates the cache context *after* the process has spawned.

In [ ]:
# Worker functions must be defined at module level to be picklable

@fleche
def expensive(x):
    return x ** 3

def _worker(x):
    """Thin wrapper that sets up a cache before calling the real function."""
    with cache(Cache(Memory({}), Memory({}))):
        return expensive(x)


with ProcessPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(_worker, range(5)))

print("Results:", results)   # [0, 1, 8, 27, 64]

### Sharing a persistent cache across processes

With in-memory storage each worker gets its own isolated cache. To share results between processes you need a **picklable, on-disk backend** (e.g. file-based storage). Pass the cache object as an argument and re-register it inside the worker:

In [ ]:
import tempfile, os
from fleche.storage.file import PickleFileStorage

tmpdir = tempfile.mkdtemp()
shared_cache = Cache(
    PickleFileStorage(os.path.join(tmpdir, "meta")),
    PickleFileStorage(os.path.join(tmpdir, "results")),
)

def _worker_shared(x, worker_cache):
    with cache(worker_cache):     # re-register the pickled cache object
        return expensive(x)


# First run — computes and persists to disk
with ProcessPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(_worker_shared, x, shared_cache) for x in range(5)]
    results = [f.result() for f in futures]

print("Results:", results)

# Verify results are in the shared cache
print("Cached:", shared_cache.contains(expensive.digest(3)))   # True

### Summary

```
ThreadPoolExecutor
  Default (no ctx.run)  →  workers use the default cache, NOT your custom one
  With ctx.run          →  workers share the same cache as the main thread ✅

ProcessPoolExecutor
  Each worker creates its own fresh cache  →  isolated per-worker caching ✅
  Pass a picklable cache as an argument   →  shared persistent cache ✅
  ctx.run                                 →  not possible (Context not picklable) ❌
```